# 01 — Exploratory Data Analysis and Preprocessing

## Automatic Fruit Classification using Computer Vision

**Dataset:** [Fruits Classification](https://www.kaggle.com/datasets/utkarshsaxenadn/fruits-classification) — Kaggle

**Classes:** Apples · Bananas · Grapes · Mangoes · Strawberries

---

## Notebook Objective

This notebook covers the first stage of the project (**Parcial 1**), focused on dataset exploration, preprocessing, and preparation for future model training.

The main objectives of this notebook are:

1. Load the dataset from Kaggle.
2. Explore the initial dataset folder structure.
3. Perform Exploratory Data Analysis (EDA).
4. Count the total number of images.
5. Count the number of images per class.
6. Visualize sample images from the dataset.
7. Analyze class balance.
8. Preprocess the images by resizing them to **224x224 pixels**.
9. Verify the preprocessing results.
10. Split the dataset into **train**, **validation**, and **test** subsets.

**Note:** This notebook is designed to run in Google Colab. At this stage, no model training is performed; the goal is to prepare the dataset for the next phase of the project.

## 2. Library Imports

In [ ]:
import os
import shutil
import random
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

## 3. Global Configuration

This section defines the main constants used throughout the notebook, including the random seed, image size, dataset split ratios, valid image extensions, and directory paths.

Using fixed configuration values helps keep the notebook organized, reproducible, and easier to modify if needed.

> **Note:** This section requires the following imports from the previous section: `random`, `pathlib.Path`

In [ ]:
# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Image preprocessing
IMAGE_SIZE = (224, 224)

# Dataset split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Validate split ratios
assert round(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 2) == 1.00, "Split ratios must sum to 1.0"

# Valid image extensions
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

# Directory paths
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
SPLIT_DIR = Path("data/split")

print("Configuration loaded successfully.")
print(f"Image size          : {IMAGE_SIZE}")
print(f"Split ratios        : train={TRAIN_RATIO} | val={VAL_RATIO} | test={TEST_RATIO}")
print(f"Random seed         : {RANDOM_SEED}")
print(f"Raw directory       : {RAW_DIR}")
print(f"Processed directory : {PROCESSED_DIR}")
print(f"Split directory     : {SPLIT_DIR}")

## 4. Kaggle Setup and Dataset Download

This section configures the Kaggle API credentials and downloads the dataset directly into Google Colab.

**Requirements:**

- A Kaggle account
- A `kaggle.json` API token

**Steps to get your `kaggle.json`:**

1. Go to [https://www.kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to the **API** section
3. Click **Create New Token**
4. A `kaggle.json` file will be downloaded to your computer

Run the cell below and upload the `kaggle.json` file when prompted.

**Note:** If this cell is executed more than once, the raw dataset folder will be cleaned before downloading the dataset again.

In [ ]:
# Install Kaggle library
!pip install kaggle -q

# Upload kaggle.json credentials
from google.colab import files

print("Upload your kaggle.json file:")
uploaded = files.upload()

# Validate uploaded file
if "kaggle.json" not in uploaded:
    raise FileNotFoundError("kaggle.json was not uploaded. Please upload your Kaggle API token file.")

# Configure Kaggle credentials
os.makedirs("/root/.config/kaggle", exist_ok=True)
!cp kaggle.json /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json

print("Kaggle credentials configured successfully.")

# Clean previous raw dataset directory if it exists
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

# Recreate folder structure
for folder in [
    RAW_DIR,
    PROCESSED_DIR,
    SPLIT_DIR / "train",
    SPLIT_DIR / "val",
    SPLIT_DIR / "test"
]:
    folder.mkdir(parents=True, exist_ok=True)

# Download and extract dataset
DATASET_SLUG = "utkarshsaxenadn/fruits-classification"

!kaggle datasets download -d {DATASET_SLUG} -p {str(RAW_DIR)} --unzip

print("Dataset downloaded and extracted successfully.")

# Show initial raw directory contents
print("\nRaw dataset folder contents:")
for item in RAW_DIR.iterdir():
    print(f"- {item.name}")

## 5. Initial Folder Exploration

This section explores the structure of the downloaded dataset to verify that the expected class folders are present and contain image files.

The goal is to identify the actual folder that contains the class directories before performing the EDA process.

In [ ]:
print("Initial raw directory contents:")
print(f"{RAW_DIR}/")

for item in sorted(RAW_DIR.iterdir()):
    item_type = "folder" if item.is_dir() else "file"
    print(f"  - {item.name} ({item_type})")


# Find folders that contain image files
candidate_class_dirs = []

for folder in RAW_DIR.rglob("*"):
    if folder.is_dir():
        image_files = [
            file for file in folder.glob("*")
            if file.suffix.lower() in IMAGE_EXTENSIONS
        ]

        if len(image_files) > 0:
            candidate_class_dirs.append(folder)


print("\nDetected folders containing images:")

if not candidate_class_dirs:
    raise FileNotFoundError("No image folders were found inside RAW_DIR.")

for class_dir in sorted(candidate_class_dirs):
    n_images = len([
        file for file in class_dir.glob("*")
        if file.suffix.lower() in IMAGE_EXTENSIONS
    ])
    print(f"  {class_dir} ({n_images} images)")


# Use detected image folders as class folders
CLASS_DIRS = sorted(candidate_class_dirs)

print("\nClass folders detected:")
for class_dir in CLASS_DIRS:
    print(f"  - {class_dir.name}")

print(f"\nTotal class folders detected: {len(CLASS_DIRS)}")

## 6. Total Image Count

This section counts the total number of images available in the dataset across all detected class folders.

This value provides a general overview of the dataset size and helps verify that the dataset was downloaded and detected correctly.

In [ ]:
# Validate that class folders were detected
if "CLASS_DIRS" not in globals() or len(CLASS_DIRS) == 0:
    raise ValueError("CLASS_DIRS is not defined or no class folders were detected. Please run the folder exploration section first.")

# Count total images across all detected class folders
total_images = sum(
    1 for class_dir in CLASS_DIRS
    for file in class_dir.glob("*")
    if file.suffix.lower() in IMAGE_EXTENSIONS
)

# Validate total image count
if total_images == 0:
    raise ValueError("No images were found in the detected class folders.")

print(f"Total images in the dataset: {total_images}")

## 7. Image Count per Class

This section counts the number of images available in each detected class folder.

The results will be stored in a dictionary and used later to create the distribution table, distribution chart, and class balance analysis.

In [ ]:
# Validate that class folders were detected
if "CLASS_DIRS" not in globals() or len(CLASS_DIRS) == 0:
    raise ValueError("CLASS_DIRS is not defined or no class folders were detected. Please run the folder exploration section first.")

# Count images per class
class_counts = {}

for class_dir in CLASS_DIRS:
    count = len([
        file for file in class_dir.glob("*")
        if file.suffix.lower() in IMAGE_EXTENSIONS
    ])
    class_counts[class_dir.name] = count

# Sort results by class name
class_counts = dict(sorted(class_counts.items()))

# Validate that all detected classes contain images
empty_classes = [class_name for class_name, count in class_counts.items() if count == 0]

if empty_classes:
    raise ValueError(f"The following classes do not contain images: {empty_classes}")

# Print results
print("Images per class:")

for class_name, count in class_counts.items():
    print(f"  {class_name:<15} : {count}")

# Validate total count consistency
sum_class_counts = sum(class_counts.values())

if "total_images" in globals():
    print(f"\nTotal images from class counts: {sum_class_counts}")
    print(f"Total images from previous section: {total_images}")

    assert sum_class_counts == total_images, "Mismatch between total_images and the sum of class_counts."
else:
    print(f"\nTotal images from class counts: {sum_class_counts}")

## 8. Distribution Table

This section displays the image count per class as a structured table, including the percentage of each class relative to the total dataset.

This table helps summarize the dataset composition and supports the class balance analysis in the following sections.

In [ ]:
# Validate required variables
if "class_counts" not in globals() or len(class_counts) == 0:
    raise ValueError("class_counts is not defined or is empty. Please run the image count per class section first.")

if "total_images" not in globals() or total_images == 0:
    raise ValueError("total_images is not defined or equal to zero. Please run the total image count section first.")

# Create distribution table
df_distribution = pd.DataFrame(
    list(class_counts.items()),
    columns=["Class", "Number of Images"]
)

# Calculate percentage per class
df_distribution["Percentage (%)"] = (
    df_distribution["Number of Images"] / total_images * 100
).round(2)

# Sort table by number of images
df_distribution = df_distribution.sort_values(
    "Number of Images",
    ascending=False
).reset_index(drop=True)

# Display table
print("Class distribution table:")
display(df_distribution)

## 9. Distribution Bar Chart

This section displays a bar chart showing the number of images per class.

The chart provides a visual overview of the class distribution and helps identify whether the dataset is balanced or imbalanced.

In [ ]:
# Validate required DataFrame
if "df_distribution" not in globals() or df_distribution.empty:
    raise ValueError("df_distribution is not defined or is empty. Please run the distribution table section first.")

# Create bar chart
plt.figure(figsize=(10, 5))

bars = plt.bar(
    df_distribution["Class"],
    df_distribution["Number of Images"],
    edgecolor="black"
)

# Add labels on top of each bar
for bar, count in zip(bars, df_distribution["Number of Images"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(count),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.title("Number of Images per Class", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Class", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Sample Image Visualization by Class

This section displays sample images from each class to visually verify the dataset quality, correct class organization, and visual variety such as color, shape, background, and lighting differences.

This visual inspection helps identify possible issues such as incorrect labels, low-quality images, inconsistent backgrounds, or corrupted files.

In [ ]:
# Validate required variables
if "CLASS_DIRS" not in globals() or len(CLASS_DIRS) == 0:
    raise ValueError("CLASS_DIRS is not defined or no class folders were detected. Please run the folder exploration section first.")

# Number of sample images to display per class
SAMPLES_PER_CLASS = 5

# Create figure
fig, axes = plt.subplots(
    len(CLASS_DIRS),
    SAMPLES_PER_CLASS,
    figsize=(15, 3 * len(CLASS_DIRS)),
    squeeze=False
)

fig.suptitle(
    "Sample Images per Class",
    fontsize=16,
    fontweight="bold",
    y=1.01
)

# Display sample images by class
for row, class_dir in enumerate(sorted(CLASS_DIRS)):
    images = [
        file for file in class_dir.glob("*")
        if file.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if len(images) == 0:
        print(f"Warning: No images found for class {class_dir.name}")
        continue

    samples = random.sample(images, min(SAMPLES_PER_CLASS, len(images)))

    for col, img_path in enumerate(samples):
        img = Image.open(img_path).convert("RGB")

        axes[row, col].imshow(img)
        axes[row, col].axis("off")

        if col == 0:
            axes[row, col].set_ylabel(
                class_dir.name,
                fontsize=11,
                fontweight="bold",
                rotation=0,
                labelpad=60,
                va="center"
            )

    # Hide unused subplot cells
    for col in range(len(samples), SAMPLES_PER_CLASS):
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 11. Corrupted Image Verification

This section attempts to open every image in the dataset to detect corrupted or unreadable files that could cause errors during preprocessing or future model training.

Detecting corrupted images at this stage helps ensure dataset quality and prevents failures in later stages of the project.

In [ ]:
# Validate required variables
if "CLASS_DIRS" not in globals() or len(CLASS_DIRS) == 0:
    raise ValueError("CLASS_DIRS is not defined or no class folders were detected. Please run the folder exploration section first.")

if "total_images" not in globals() or total_images == 0:
    raise ValueError("total_images is not defined or equal to zero. Please run the total image count section first.")

# Verify image files
corrupted_images = []

for class_dir in CLASS_DIRS:
    for img_path in class_dir.glob("*"):
        if img_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception as e:
            corrupted_images.append({
                "file": str(img_path),
                "class": class_dir.name,
                "error": str(e)
            })

# Display results
if corrupted_images:
    print(f"Corrupted or unreadable images found: {len(corrupted_images)}")
    df_corrupted = pd.DataFrame(corrupted_images)
    display(df_corrupted)
else:
    print(f"No corrupted images found. All {total_images} images are readable.")

## 12. Class Balance Analysis

This section analyzes whether the dataset is balanced or imbalanced by comparing the number of images across classes.

A dataset is considered balanced when all classes have a similar number of images. An imbalanced dataset can cause a model to perform better on majority classes and worse on minority classes.

The imbalance ratio is calculated as:

```
largest_class_count / smallest_class_count
```

A lower ratio indicates a more balanced dataset.

In [ ]:
# Validate required variables
if "class_counts" not in globals() or len(class_counts) == 0:
    raise ValueError("class_counts is not defined or is empty. Please run the image count per class section first.")

if any(count == 0 for count in class_counts.values()):
    raise ValueError("At least one class has zero images. Balance analysis cannot be performed.")

# Identify largest and smallest classes
max_class = max(class_counts, key=class_counts.get)
min_class = min(class_counts, key=class_counts.get)

max_count = class_counts[max_class]
min_count = class_counts[min_class]

# Calculate imbalance ratio
imbalance_ratio = round(max_count / min_count, 2)

print("Class Balance Analysis")
print("=" * 40)
print(f"Largest class   : {max_class} ({max_count} images)")
print(f"Smallest class  : {min_class} ({min_count} images)")
print(f"Imbalance ratio : {imbalance_ratio}x")
print()

# Interpret imbalance ratio
if imbalance_ratio < 1.5:
    balance_status = "BALANCED"
    recommendation = "The dataset has a similar number of images across classes."
elif imbalance_ratio < 3.0:
    balance_status = "MODERATE IMBALANCE"
    recommendation = "It is recommended to monitor performance per class during future model training."
else:
    balance_status = "IMBALANCED"
    recommendation = "Balancing techniques such as class weights, oversampling, or data augmentation should be considered."

print(f"Conclusion: The dataset is {balance_status}.")
print(recommendation)

## 13. Resize Images to 224x224

This section resizes all images from the raw dataset to **224x224 pixels** and saves them to the processed directory while preserving the class folder structure.

The target size of **224x224** is commonly used in Computer Vision and is compatible with many Transfer Learning architectures such as VGG16, ResNet50, MobileNet, and EfficientNet.

All images are converted to RGB format before resizing to ensure consistency across the dataset.

If this cell is executed more than once, the processed directory is cleaned before generating the resized images again.

In [ ]:
# Validate required variables
if "CLASS_DIRS" not in globals() or len(CLASS_DIRS) == 0:
    raise ValueError("CLASS_DIRS is not defined or no class folders were detected. Please run the folder exploration section first.")

if "IMAGE_SIZE" not in globals():
    raise ValueError("IMAGE_SIZE is not defined. Please run the global configuration section first.")

# Clean processed directory before resizing
if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Resize images
resize_errors = []
total_processed = 0

for class_dir in CLASS_DIRS:
    output_class_dir = PROCESSED_DIR / class_dir.name
    output_class_dir.mkdir(parents=True, exist_ok=True)

    image_files = [
        file for file in class_dir.glob("*")
        if file.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for img_path in image_files:
        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
                img = img.resize(IMAGE_SIZE, Image.LANCZOS)
                img.save(output_class_dir / img_path.name)

            total_processed += 1

        except Exception as e:
            resize_errors.append({
                "file": str(img_path),
                "class": class_dir.name,
                "error": str(e)
            })

# Display results
print("Image resizing completed.")
print(f"Images processed successfully : {total_processed}")
print(f"Errors                        : {len(resize_errors)}")

if "total_images" in globals():
    print(f"Expected total images         : {total_images}")

    if total_processed == total_images:
        print("All images were processed successfully.")
    else:
        print("Warning: The number of processed images does not match the expected total.")

if resize_errors:
    df_resize_errors = pd.DataFrame(resize_errors)
    display(df_resize_errors)

## 14. Resize Verification

This section verifies that all processed images have the correct dimensions of **224x224 pixels**.

The expected result is that all images share a single size entry of `(224, 224)`. Any other size found indicates a preprocessing error.

This verification confirms that the preprocessing step was applied correctly before splitting the dataset.

In [ ]:
# Validate processed directory
if not PROCESSED_DIR.exists():
    raise FileNotFoundError("PROCESSED_DIR does not exist. Please run the resize section first.")

# Verify processed image sizes
size_counts = {}
verify_errors = []

processed_images = [
    img_path for img_path in PROCESSED_DIR.rglob("*")
    if img_path.suffix.lower() in IMAGE_EXTENSIONS
]

if len(processed_images) == 0:
    raise ValueError("No processed images were found. Please run the resize section first.")

for img_path in processed_images:
    try:
        with Image.open(img_path) as img:
            size = img.size
            size_counts[size] = size_counts.get(size, 0) + 1
    except Exception as e:
        verify_errors.append({
            "file": str(img_path),
            "error": str(e)
        })

# Display image sizes found
print("Image sizes found in processed directory:")

for size, count in sorted(size_counts.items()):
    status = "OK" if size == IMAGE_SIZE else "ERROR"
    print(f"  {size} : {count} images [{status}]")

# Display reading errors if any
if verify_errors:
    print(f"\nErrors reading {len(verify_errors)} images:")
    df_verify_errors = pd.DataFrame(verify_errors)
    display(df_verify_errors)

# Final verification
resize_verification_passed = (
    len(size_counts) == 1 and
    IMAGE_SIZE in size_counts and
    len(verify_errors) == 0
)

if resize_verification_passed:
    print(f"\nVerification passed. All images are {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]} pixels.")
else:
    print("\nVerification failed. Some images have incorrect dimensions or could not be read.")

## 15. Train / Validation / Test Split

This section splits the processed dataset into three subsets: **train**, **validation**, and **test**.

| Subset | Ratio | Purpose |
|---|---:|---|
| Train | 70% | Used to train the model |
| Validation | 15% | Used to monitor performance during training |
| Test | 15% | Used to evaluate the final model |

The split is performed **per class** to preserve the class distribution across all subsets.

A fixed random seed is used to ensure reproducibility.

If this cell is executed more than once, the split directory is cleaned before generating the new split.

In [ ]:
# Validate processed directory
if not PROCESSED_DIR.exists():
    raise FileNotFoundError("PROCESSED_DIR does not exist. Please run the resize section first.")

# Validate split ratios
assert round(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 2) == 1.00, "Split ratios must sum to 1.0"

# Get processed class folders
processed_class_dirs = [
    class_dir for class_dir in sorted(PROCESSED_DIR.iterdir())
    if class_dir.is_dir()
]

if len(processed_class_dirs) == 0:
    raise ValueError("No processed class folders were found. Please run the resize section first.")

# Clean split directory before splitting
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

# Create split directories
for subset in ["train", "val", "test"]:
    (SPLIT_DIR / subset).mkdir(parents=True, exist_ok=True)

# Split dataset per class
split_log = []

for class_dir in processed_class_dirs:
    images = [
        file for file in class_dir.glob("*")
        if file.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if len(images) < 3:
        raise ValueError(
            f"Class '{class_dir.name}' has fewer than 3 images. "
            "At least 3 images are required to split into train, validation, and test."
        )

    train_imgs, temp_imgs = train_test_split(
        images,
        test_size=(1 - TRAIN_RATIO),
        random_state=RANDOM_SEED,
        shuffle=True
    )

    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=RANDOM_SEED,
        shuffle=True
    )

    subsets = {
        "train": train_imgs,
        "val": val_imgs,
        "test": test_imgs
    }

    for subset_name, subset_imgs in subsets.items():
        destination_dir = SPLIT_DIR / subset_name / class_dir.name
        destination_dir.mkdir(parents=True, exist_ok=True)

        for img_path in subset_imgs:
            shutil.copy2(img_path, destination_dir / img_path.name)

    split_log.append({
        "Class": class_dir.name,
        "Train": len(train_imgs),
        "Validation": len(val_imgs),
        "Test": len(test_imgs),
        "Total": len(images)
    })

    print(
        f"{class_dir.name:<15} "
        f"train={len(train_imgs)} | "
        f"val={len(val_imgs)} | "
        f"test={len(test_imgs)} | "
        f"total={len(images)}"
    )

# Create split summary table
df_split_log = pd.DataFrame(split_log)

print("\nDataset split completed successfully.")
display(df_split_log)

## 16. Count Images per Subset

This section counts the total number of images in each subset after the split and verifies that the total matches the number of processed images.

This validation ensures that no images were lost during the train, validation, and test split process.

In [ ]:
# Validate split directory
if not SPLIT_DIR.exists():
    raise FileNotFoundError("SPLIT_DIR does not exist. Please run the split section first.")

# Validate expected subset folders
expected_subsets = ["train", "val", "test"]

for subset in expected_subsets:
    subset_path = SPLIT_DIR / subset
    if not subset_path.exists():
        raise FileNotFoundError(f"Subset folder '{subset}' does not exist. Please run the split section first.")

# Count images per subset
subset_counts = {}

for subset in expected_subsets:
    subset_path = SPLIT_DIR / subset
    total = sum(
        1 for file in subset_path.rglob("*")
        if file.suffix.lower() in IMAGE_EXTENSIONS
    )
    subset_counts[subset] = total

# Validate that images were found
total_after_split = sum(subset_counts.values())

if total_after_split == 0:
    raise ValueError("No images were found after the split process.")

# Create summary table
df_subset_counts = pd.DataFrame([
    {
        "Subset": subset,
        "Images": count,
        "Percentage (%)": round(count / total_after_split * 100, 2)
    }
    for subset, count in subset_counts.items()
])

# Display results
print("Images per subset:")
display(df_subset_counts)

print(f"\nTotal images after split: {total_after_split}")

# Verify total consistency
split_total_verified = False

if "total_processed" in globals():
    print(f"Total processed images : {total_processed}")
    if total_after_split == total_processed:
        split_total_verified = True
        print("Split verified. Total matches the number of processed images.")
    else:
        print("Warning: Total after split does not match the number of processed images.")

elif "total_images" in globals():
    print(f"Original total images : {total_images}")
    if total_after_split == total_images:
        split_total_verified = True
        print("Split verified. Total matches the original dataset count.")
    else:
        print("Warning: Total after split does not match the original dataset count.")

else:
    print("Total verification skipped because no previous total count was found.")

## 17. Count Images per Class per Subset

This section counts the number of images per class in each subset and displays the results as a pivot table.

This verifies that the class distribution is correctly preserved across the **train**, **validation**, and **test** subsets after the split.

The resulting table helps confirm that each class is represented in all subsets.

In [ ]:
# Validate split directory
if not SPLIT_DIR.exists():
    raise FileNotFoundError("SPLIT_DIR does not exist. Please run the split section first.")

# Validate expected subset folders
expected_subsets = ["train", "val", "test"]

for subset in expected_subsets:
    subset_path = SPLIT_DIR / subset
    if not subset_path.exists():
        raise FileNotFoundError(f"Subset folder '{subset}' does not exist. Please run the split section first.")

# Count images per class per subset
records = []

for subset in expected_subsets:
    subset_path = SPLIT_DIR / subset

    for class_dir in sorted(subset_path.iterdir()):
        if not class_dir.is_dir():
            continue

        count = len([
            file for file in class_dir.glob("*")
            if file.suffix.lower() in IMAGE_EXTENSIONS
        ])

        records.append({
            "Subset": subset,
            "Class": class_dir.name,
            "Images": count
        })

# Validate records
if len(records) == 0:
    raise ValueError("No class images were found in the split directory.")

# Create DataFrame
df_class_per_subset = pd.DataFrame(records)

# Create pivot table
pivot_class_split = df_class_per_subset.pivot_table(
    index="Class",
    columns="Subset",
    values="Images",
    aggfunc="sum",
    fill_value=0
)

# Ensure column order
pivot_class_split = pivot_class_split[expected_subsets]

# Add total per class
pivot_class_split["Total"] = pivot_class_split.sum(axis=1)

# Add total row
pivot_class_split.loc["Total"] = pivot_class_split.sum(axis=0)

# Display pivot table
print("Images per class per subset:")
display(pivot_class_split)

## 18. Final EDA Summary

This section prints a complete summary of the results obtained during the exploratory data analysis and preprocessing stages.

The summary includes:

- Dataset information
- Total number of images
- Number of classes
- Image count per class
- Class balance analysis
- Image preprocessing results
- Resize verification
- Train / validation / test split results

This summary can be used to update the README file with the final results from Parcial 1.

In [ ]:
# Validate required variables before generating the final summary
required_variables = [
    "total_images",
    "class_counts",
    "balance_status",
    "imbalance_ratio",
    "IMAGE_SIZE",
    "resize_verification_passed",
    "subset_counts",
    "split_total_verified"
]

missing_variables = [
    var for var in required_variables
    if var not in globals()
]

if missing_variables:
    raise ValueError(
        f"The following required variables are missing: {missing_variables}. "
        "Please run all previous sections before generating the final summary."
    )

# Print final EDA summary
print("=" * 60)
print("FINAL EDA SUMMARY")
print("=" * 60)

print("Dataset Information")
print("-" * 60)
print("Dataset           : Fruits Classification")
print("Source            : Kaggle")
print("Task type         : Multiclass image classification")
print(f"Total images      : {total_images}")
print(f"Number of classes : {len(class_counts)}")
print(f"Classes           : {', '.join(class_counts.keys())}")

print("\nClass Distribution")
print("-" * 60)

for cls, count in class_counts.items():
    percentage = round(count / total_images * 100, 2)
    print(f"{cls:<15} : {count:<6} images ({percentage}%)")

print("\nClass Balance")
print("-" * 60)
print(f"Balance status    : {balance_status}")
print(f"Imbalance ratio   : {imbalance_ratio}x")

print("\nPreprocessing")
print("-" * 60)
print(f"Image size         : {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]} pixels")
print("Color format       : RGB")
print(f"Resize verification: {'Passed' if resize_verification_passed else 'Failed'}")

print("\nDataset Split")
print("-" * 60)

total_after_split = sum(subset_counts.values())

for subset, count in subset_counts.items():
    percentage = round(count / total_after_split * 100, 2)
    print(f"{subset:<15} : {count:<6} images ({percentage}%)")

print(f"\nSplit verified     : {'Yes' if split_total_verified else 'No'}")

print("=" * 60)
print("Dataset preparation for Parcial 1 has been completed successfully.")
print("=" * 60)

## 19. Conclusions

This section summarizes the main findings and conclusions from the exploratory data analysis and preprocessing process carried out in this notebook.

---

During the exploratory data analysis, the dataset was found to contain a total of **[TOTAL_IMAGES]** images distributed across **5 classes**:

- Apples
- Bananas
- Grapes
- Mangoes
- Strawberries

The class distribution analysis showed that the dataset is **[BALANCED / MODERATELY IMBALANCED / IMBALANCED]**, with an imbalance ratio of **[IMBALANCE_RATIO]x** between the largest and smallest class.

This information will be relevant when defining the training strategy in the next stage, since class imbalance can affect model performance and may require techniques such as data augmentation, class weights, or additional evaluation per class.

Sample images from each class were visually inspected to verify the dataset quality, correct class organization, and the presence of visual variety such as color, shape, background, and lighting differences.

All images were converted to **RGB format** and resized to **224x224 pixels**, standardizing the input format for future Computer Vision classification work.

The resize verification confirmed that **[RESIZE_VERIFICATION_RESULT]**.

The dataset was split into three subsets using a **70% / 15% / 15%** ratio:

| Subset | Percentage | Purpose |
|---|---:|---|
| Train | 70% | Used to train the model |
| Validation | 15% | Used to monitor performance during training |
| Test | 15% | Used to evaluate the final model |

The split was performed per class to preserve the class distribution across all subsets.

The split verification confirmed that **[SPLIT_VERIFICATION_RESULT]**.

---

The dataset is now fully prepared for the next course stage, where classification models can be built, trained, and evaluated.